# Standard Convolution
- Conv2d: Sum of X_c * W_k, c + b_k = Local correlation + Bias + Weight sharing (c is channel)
- Parameters = C_out * C_in * H_K * W_K + C_out (learns only filters, which is C_out * C_in * H_K * W_K)
- In: B x C_in x H_in x W_in
- Out: B x C_out x H_out x W_out
- Out (H, W) = floor((In + 2Pad - Dilation * (Kernel - 1) - 1) / Stride) + 1 = floor((In + 2Pad - Kernel) / Stride) + 1
- Calculate receptive field: Watch each output value depends on how many values from the nearest layer
- Formula: RF = k_1 + (k_2 - 1) * s_1

In [ ]:
import torch
import torch.nn as nn

class Conv2D(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1, bias=True)

    def forward(self, x):
        return self.conv(x)

# Depthwise Convolution
- Strictly 1 - 1 between C_in and C_out
- In: B x C_in x H_in x W_in
- Out: B x C_in x H_out x W_out
- Parallel computing
- Parameter = C_in * H_K * W_K + C_in
- Same receptive field, but do not accumulate information between channel

In [ ]:
import torch
import torch.nn as nn

class DepthwiseConv2D(nn.Module):
    def __init__(self):
        super().__init__()
        # Set groups for depthwise convolution
        self.depthwise_conv = nn.Conv2d(in_channels=3, out_channels=3, kernel_size=3, stride=1, padding=1, groups=3, bias=True)

    def forward(self, x):
        return self.depthwise_conv(x)

# Pointwise Convolution
- 1 x 1 convolution kernel
- Mix channel
- In: B x C_in x H x W
- Out: B x C_out x H x W
- Parameter: C_out x C_in + C_out
- Meaning: Recombine channels at each pixel

In [ ]:
import torch
import torch.nn as nn

class PointwiseConv2D(nn.Module):
    def __init__(self):
        super().__init__()
        # Pointwise convolution with kernel size 1
        self.pointwise_conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=1, stride=1, padding=0, bias=True)

    def forward(self, x):
        return self.pointwise_conv(x)

# Depthwise separable convolution
- = Depthwise + Pointwise
- Compare to standard Conv2D:
+ Standard: C_out x C_in x H_K x W_K + C_out
+ Depthwise seperable: (C_in x H_K x W_K + C_in) + (C_out x C_in + C_out) <<< Standard

In [ ]:
import torch
import torch.nn as nn

class DepthwiseSeparableConv2D(nn.Module):
    def __init__(self):
        super().__init__()
        # Depthwise convolution
        self.depthwise_conv = nn.Conv2d(in_channels=3, out_channels=3, kernel_size=3, stride=1, padding=1, groups=3, bias=True)
        # Pointwise convolution
        self.pointwise_conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=1, stride=1, padding=0, bias=True)

    def forward(self, x):
        x = self.depthwise_conv(x)
        x = self.pointwise_conv(x)
        return x

In [ ]:
import torch
import torch.nn as nn

class DepthwiseSeperableConv2D(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=3, kernel_size=3, stride=1, padding=1, groups=3, bias=True),  # Depthwise
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=1, stride=1, padding=0, bias=True)  # Pointwise
        )

    def forward(self, x):
        return self.layers(x)

# Dilation
- Wider context range without smaller grid

In [ ]:
import torch
import torch.nn as nn

class ConvDilation2D(nn.Module):
    def __init__(self):
        super().__init__()
        # With dilation, the effective kernel size increases, allowing for a larger receptive field without increasing the number of parameters
        self.conv_dilation = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=2, dilation=2, bias=True)

    def forward(self, x):
        return self.conv_dilation(x)